# DECODER-ONLY FOR SEQUENCE GENERATION

The model learns to predict both YEAR_n and event tokens.
Now, years are numerical and have their own embedding. 

In [89]:
import numpy as np
import pandas as pd


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [118]:
# read training data, create main main_vocab from all events in the data, create year_main_vocab from 0 to 110
df = pd.read_csv("Event_data.csv")

In [119]:
# create main_vocabulary from all diff words in df.event
event_tokens = [c for c in set(df.event)]
main_vocab = {token: idx for idx, token in enumerate(event_tokens)}
id_to_token = {idx: token for token, idx in main_vocab.items()}

In [120]:
class LifeEventDatasetFlat(Dataset):
    def __init__(self, df, main_vocab, max_age=120):
        self.main_vocab = main_vocab
        self.sequences = []

        assert {'patid', 'age', 'event'}.issubset(df.columns)

        grouped = df.groupby('patid')
        for _, group in grouped:
            group = group.sort_values('age')

            event_ids = []
            age_vals = []

            for _, row in group.iterrows():
                age = int(row['age'])
                event = row['event']

                if event in main_vocab and 0 <= age <= max_age:
                    event_ids.append(main_vocab[event])
                    age_vals.append(age)
                else:
                    raise ValueError(f"Unknown event '{event}' or age '{age}' out of bounds.")

            self.sequences.append((torch.tensor(event_ids, dtype=torch.long),
                                   torch.tensor(age_vals, dtype=torch.long)))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


In [121]:
class SimpleTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, max_seq_len=256, max_age=120):
        super().__init__()
        self.event_embedding = nn.Embedding(vocab_size, d_model)
        self.age_embedding = nn.Embedding(max_age + 1, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)

        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, event_ids, age_vals):
        B, L = event_ids.size()

        # Get embeddings
        event_emb = self.event_embedding(event_ids)
        age_emb = self.age_embedding(age_vals)
        pos_emb = self.pos_embedding(torch.arange(L, device=event_ids.device).unsqueeze(0).expand(B, L))

        x = event_emb + age_emb + pos_emb  # total embedding

        # Transformer forward
        x = x.transpose(0, 1)  # (L, B, D)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(L).to(x.device)
        x = self.transformer(x, x, tgt_mask=tgt_mask)
        x = x.transpose(0, 1)  # (B, L, D)

        return self.output_head(x)  # (B, L, vocab_size)


In [122]:
def collate_fn(batch):
    from torch.nn.utils.rnn import pad_sequence

    events, ages = zip(*batch)
    padded_events = pad_sequence(events, batch_first=True, padding_value=0)
    padded_ages = pad_sequence(ages, batch_first=True, padding_value=0)

    return padded_events, padded_ages


## TRAINING THE MODEL 

In [123]:
# --- Hyperparameters ---
batch_size = 5
num_epochs = 100
lr = 1e-3
pad_token_id = 0  # You must ensure this ID is reserved for padding in your main_vocab


In [124]:

# --- Dataset and DataLoader ---
dataset = LifeEventDatasetFlat(df, main_vocab)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)


In [126]:
del(model)
# --- Model, Loss, Optimizer ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleTransformerLM(vocab_size=len(main_vocab)).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=pad_token_id)
optimizer = optim.Adam(model.parameters(), lr=lr)


In [127]:

# --- Training Loop ---
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for event_ids, age_vals in dataloader:
        event_ids = event_ids.to(device)       # shape: (B, L)
        age_vals = age_vals.to(device)         # shape: (B, L)

        optimizer.zero_grad()

        # Inputs are everything except the last token
        input_event = event_ids[:, :-1]
        input_age = age_vals[:, :-1]

        # Targets are everything except the first token
        target = event_ids[:, 1:]

        output = model(input_event, input_age)  # (B, L-1, vocab_size)

        # Reshape for loss: (B*(L-1), vocab_size) vs. (B*(L-1))
        loss = criterion(output.reshape(-1, output.size(-1)), target.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    if (epoch%10 == 0):
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")


Epoch 1/100 | Loss: 1.3944
Epoch 11/100 | Loss: 0.0100
Epoch 21/100 | Loss: 0.0039
Epoch 31/100 | Loss: 0.0029
Epoch 41/100 | Loss: 0.0025
Epoch 51/100 | Loss: 0.0020
Epoch 61/100 | Loss: 0.0017
Epoch 71/100 | Loss: 0.0014
Epoch 81/100 | Loss: 0.0012
Epoch 91/100 | Loss: 0.0012


### GENERATOR OF NEW SEQUENCES
* Top-k sampling (randomly from top-k logits)
* Temperature scaling (controls randomness/sharpness)



In [128]:
def generate_sequences_batch(
    model,
    start_events,        # list of event tokens (e.g., ["born", "university"])
    start_ages,          # list of age ints (e.g., [0, 28])
    vocab,
    id_to_token,
    max_age=120,
    max_length=50,
    stop_token="died",
    device="cpu",
    k=5,
    temperature=1.0,
    num_sequences=5
):
    model.eval()
    
    input_event_ids = [vocab[tok] for tok in start_events]
    input_age_vals = [min(int(age), max_age) for age in start_ages]

    sequences = []

    for _ in range(num_sequences):
        event_ids = input_event_ids.copy()
        age_vals = input_age_vals.copy()

        for _ in range(max_length):
            input_event = torch.tensor(event_ids, dtype=torch.long).unsqueeze(0).to(device)
            input_age = torch.tensor(age_vals, dtype=torch.long).unsqueeze(0).to(device)

            with torch.no_grad():
                logits = model(input_event, input_age)  # (1, L, vocab_size)
                next_token_logits = logits[0, -1] / temperature

                topk = torch.topk(next_token_logits, k=min(k, logits.size(-1)))
                indices = topk.indices
                probs = torch.ones_like(indices, dtype=torch.float) / len(indices)
                next_token_id = indices[torch.multinomial(probs, 1).item()].item()

            next_token = id_to_token[next_token_id]
            next_age = min(age_vals[-1] + 1, max_age)

            event_ids.append(next_token_id)
            age_vals.append(next_age)

            if next_token == stop_token:
                break

        # Combine tokens and age values
        sequence = [(id_to_token[tok_id], age) for tok_id, age in zip(event_ids, age_vals)]
        sequences.append(sequence)

    return sequences



In [132]:
sequences = generate_sequences_batch(
    model,
    start_events=["born", "flu"],
    start_ages=[0, 28],
    vocab= main_vocab,
    id_to_token=id_to_token,
    num_sequences=10,
    k=3,
    temperature=0.7
)

for i, seq in enumerate(sequences):
    formatted = ", ".join(f"{age} {token}" for token, age in seq)
    print(f"Sequence {i+1}: {formatted}")

Sequence 1: 0 born, 28 flu, 29 died
Sequence 2: 0 born, 28 flu, 29 died
Sequence 3: 0 born, 28 flu, 29 born, 30 flu, 31 flu, 32 born, 33 died
Sequence 4: 0 born, 28 flu, 29 died
Sequence 5: 0 born, 28 flu, 29 born, 30 born, 31 died
Sequence 6: 0 born, 28 flu, 29 flu, 30 born, 31 born, 32 flu, 33 died
Sequence 7: 0 born, 28 flu, 29 born, 30 flu, 31 born, 32 flu, 33 born, 34 flu, 35 flu, 36 died
Sequence 8: 0 born, 28 flu, 29 died
Sequence 9: 0 born, 28 flu, 29 flu, 30 born, 31 died
Sequence 10: 0 born, 28 flu, 29 died


In [133]:
# convert generated data back to the dataframe 
rows = []
for i, seq in enumerate(sequences, start=1):
    for token, age in seq:
        rows.append({"sequence": i, "age": age, "event": token})
df_gen = pd.DataFrame(rows)

In [134]:
df_gen.head()

,sequence,age,event
0,1,0,born
1,1,28,flu
2,1,29,died
3,2,0,born
4,2,28,flu
